# Capstone — Position-Adjusted CTR Opportunity Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreeyeshbaral/ShreeyeshAssignment1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook mirrors the deployed research paper (*Lane 4 — FlyRank Capstone*).
It implements the full pipeline: research question framing, data safety & leakage checks,
rule baseline score, Random Forest model training, client-grouped validation audit,
ranked action queue with reason codes, artifacts generation, **5-minute showcase demo outline**,
and **shareable cuts (social post + employer summary)**.

**Language standard:** Public-safe language everywhere — observed, measured, directional, decision-support.

## 1. Question

**FlyRank Problem Context:** FlyRank builds content as infrastructure — researching, publishing, and optimizing thousands of content assets across client portfolios. FlyRank's product suite uses hand-written heuristic flags (e.g. static CTR thresholds like `CTR < 0.5%`) to flag underperforming content. However, fixed threshold rules flag 9,759 pages in this dataset — creating 195 review cycles for editors operating at 50 pages/cycle — while ignoring position context (position 3 vs 18).

**Research Question:**
> *Which visible pages in a published portfolio under-capture clicks or engagement relative to their position tier, and should be reviewed first for title, metadata, or content refresh?*

**Unit of Analysis:** Page (anonymized content ID).
**Output:** Ranked action queue with reason codes.
**Action:** Content editor audits titles/meta descriptions (CTR gap) or content structure (engagement gap).
**Cost of Errors:** False positive = 15-30 min wasted editor time. False negative = compounding wasted search visibility.

In [1]:
# ── Load Data & Define Working Slice ────────────────────────────────────────
import pandas as pd
import numpy as np
import os, json, pathlib, warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
np.random.seed(SEED)

# Load data
local_path = '../../data/raw/content_refresh_anonymized.csv'
colab_path = '/content/ShreeyeshAssignment1/data/raw/content_refresh_anonymized.csv'
csv_path = local_path if os.path.exists(local_path) else colab_path
df = pd.read_csv(csv_path)

# Lane 4 working slice: visible pages with sufficient search volume
lane4 = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 100)].copy()

# Proxy label: is_under_ctr = 1 if CTR < median CTR of position tier
tier_median = lane4.groupby('position_tier')['ctr'].median()
lane4['tier_median_ctr'] = lane4['position_tier'].map(tier_median.to_dict())
lane4['ctr_gap'] = lane4['tier_median_ctr'] - lane4['ctr']
lane4['is_under_ctr'] = (lane4['ctr'] < lane4['tier_median_ctr']).astype(int)

base_rate = lane4['is_under_ctr'].mean()
print(f"Total raw rows: {len(df):,}")
print(f"Lane 4 working slice: {len(lane4):,} rows ({len(lane4)/len(df):.1%})")
print(f"Base rate (is_under_ctr = 1): {base_rate:.1%}")

## 2. Data & Data Safety

**Source:** FlyRank ML Internship dataset (79-million-row warehouse on Hugging Face). Bundled 30,000-row starter sample across 32 anonymized clients.

**Excluded Fields (Leakage Prevention):**
- `ctr`, `clicks_90d`, `clicks_last_30d`, `clicks_prev_30d`: Direct target components.
- `trend_direction`, `trend_pct`: Label-adjacent thresholded fields.
- `client_id`: Used strictly for client grouping in split design, never as a feature.

In [2]:
# ── Feature Setup & Data Preprocessing ──────────────────────────────────────
NUMERIC_FEATURES = [
    'avg_position', 'impressions_90d', 'days_since_last_update',
    'word_count', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'content_age_days', 'days_with_impressions', 'days_with_sessions',
    'pageviews_90d', 'sessions_90d',
]
CATEGORICAL_FEATURES = ['content_type', 'main_intent', 'position_tier', 'freshness_tier']
TARGET = 'is_under_ctr'

# Feature engineering
lane4['has_word_count'] = lane4['word_count'].notna().astype(int)
for col in NUMERIC_FEATURES:
    lane4[col] = lane4[col].fillna(0)
for col in CATEGORICAL_FEATURES:
    lane4[col] = lane4[col].fillna('unknown')

lane4_encoded = pd.get_dummies(lane4, columns=CATEGORICAL_FEATURES, drop_first=False)
ohe_cols = [c for c in lane4_encoded.columns if any(c.startswith(f'{cat}_') for cat in CATEGORICAL_FEATURES)]
feature_cols = NUMERIC_FEATURES + ['has_word_count'] + sorted(ohe_cols)

print(f"Total model features: {len(feature_cols)} ({len(NUMERIC_FEATURES)} numeric, 1 engineered, {len(ohe_cols)} OHE categorical)")

## 3. Methodology & Baseline

**Validation Design:** 30-client GroupShuffleSplit (8 clients held out completely). Prevents portfolio leakage across clients.

**Baseline Score Formula:**
`score = visible * actionable * log1p(impressions_90d) * (1 + stale)`
Where `visible = (impressions_90d >= 500)`, `actionable = (3 < avg_position <= 20)`, `stale = (days_since_last_update >= 90)`.

**Model:** Random Forest Classifier (`n_estimators=200`, `max_depth=5`, `class_weight='balanced'`, `random_state=42`).

In [3]:
# ── Grouped Evaluation: Model vs Baseline ──────────────────────────────────
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(gss.split(lane4_encoded, lane4_encoded[TARGET], groups=lane4_encoded['client_id']))

train_df = lane4_encoded.iloc[train_idx]
test_df = lane4_encoded.iloc[test_idx]
test_raw = lane4.iloc[test_idx].copy()

X_train, y_train = train_df[feature_cols], train_df[TARGET].values
X_test, y_test = test_df[feature_cols], test_df[TARGET].values

# 1. Rule Baseline Score
test_raw['b_visible'] = (test_raw['impressions_90d'] >= 500).astype(int)
test_raw['b_actionable'] = ((test_raw['avg_position'] > 3) & (test_raw['avg_position'] <= 20)).astype(int)
test_raw['b_stale'] = (test_raw['days_since_last_update'] >= 90).astype(int)
test_raw['rule_score'] = test_raw['b_visible'] * test_raw['b_actionable'] * np.log1p(test_raw['impressions_90d']) * (1 + test_raw['b_stale'])

# 2. Random Forest Model
rf_eval = RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=SEED, n_jobs=-1)
rf_eval.fit(X_train, y_train)
test_raw['model_score'] = rf_eval.predict_proba(X_test)[:, 1]

# Precision@K calculation
def precision_at_k(df_sub, score_col, k):
    top_k = df_sub.sort_values(score_col, ascending=False).head(k)
    return top_k['is_under_ctr'].mean()

metrics = {
    'Baseline': {
        'AUC': float(roc_auc_score(y_test, test_raw['rule_score'])),
        'P@10': float(precision_at_k(test_raw, 'rule_score', 10)),
        'P@20': float(precision_at_k(test_raw, 'rule_score', 20)),
        'P@50': float(precision_at_k(test_raw, 'rule_score', 50))
    },
    'Random Forest': {
        'AUC': float(roc_auc_score(y_test, test_raw['model_score'])),
        'P@10': float(precision_at_k(test_raw, 'model_score', 10)),
        'P@20': float(precision_at_k(test_raw, 'model_score', 20)),
        'P@50': float(precision_at_k(test_raw, 'model_score', 50))
    }
}

metrics_df = pd.DataFrame(metrics).T
print("=== Grouped Test Fold Evaluation (8 Held-Out Clients, 4,610 Rows) ===")
print(metrics_df.round(2))

## 4. Results (vs Baseline)

**Headline Result:** On the 8 held-out client portfolios, Random Forest achieved **Precision@50 = 0.76** (38/50 true under-performers) vs **Precision@50 = 0.40** (20/50) for the rule baseline — an observed **1.9× lift in queue precision**.

**AUC:** Random Forest AUC = 0.72 vs Rule Baseline AUC = 0.47 (below random).

In [4]:
# ── Permutation Importance on Test Set ──────────────────────────────────────
from sklearn.inspection import permutation_importance

result = permutation_importance(rf_eval, X_test, y_test, scoring='roc_auc', n_repeats=10, random_state=SEED)
perm_importances = pd.Series(result.importances_mean, index=feature_cols).sort_values(ascending=False)

print("Top 5 Feature Importances (AUC Drop on Held-Out Test Fold):")
for feat, val in perm_importances.head(5).items():
    print(f"  {feat:25s}: {val:.4f}")

## 5. Limitations & Honest Framing

**What this work CANNOT claim:**
1. **Observational, not causal:** We observe signals associated with CTR gap. We cannot claim updating a title tag will cause CTR to rise without an A/B experiment.
2. **Proxy label:** `is_under_ctr` compares CTR against position-tier median in a snapshot, not longitudinal CTR recovery.
3. **Single grouped split:** Tested on one split of 30 clients. Requires forward temporal validation before operational deployment.
4. **No causal claims about Google's algorithm:** We do not claim to have 'predicted Google's algorithm'.

In [5]:
# ── Build Full Queue & Assign Reason Codes ──────────────────────────────────
# Fit RF on all data for production queue scoring
rf_full = RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=SEED, n_jobs=-1)
rf_full.fit(lane4_encoded[feature_cols], lane4_encoded[TARGET].values)
lane4['model_score'] = rf_full.predict_proba(lane4_encoded[feature_cols])[:, 1]

# Compute data thresholds for reason codes
p25_days_sessions = lane4['days_with_sessions'].quantile(0.25)
med_engagement = lane4['engagement_rate'].median()

def assign_reason_code(row):
    score = row['model_score']
    if score <= 0.5:
        return 'no_action_needed'
    if row['impressions_90d'] >= 1000 and score > 0.7:
        return 'high_volume_underperformer'
    if row['avg_position'] > 10 and row['avg_position'] <= 20 and row['days_since_last_update'] >= 90:
        return 'position_decay_risk'
    if row['days_with_sessions'] <= p25_days_sessions:
        return 'low_session_consistency'
    return 'general_ctr_opportunity'

lane4['reason_code'] = lane4.apply(assign_reason_code, axis=1)
print("=== Reason Code Queue Summary (22,006 Pages) ===")
print(lane4['reason_code'].value_counts())
print(f"Total flagged pages (model_score > 0.5): {(lane4['model_score'] > 0.5).sum():,}")

## 6. Ranked Recommendations

**Action Playbook Priority:**
1. **High-Volume Underperformers (342 pages) — Act First:** Audit title and meta description; highest absolute click return per editor hour.
2. **Position Decay Risk (971 pages) — Protect Visibility:** Content freshness check and internal link audit for striking-distance pages (positions 11-20).
3. **Low Session Consistency (3,884 pages) — Investigate Engagement:** Audit on-page experience, load speed, and readability.
4. **General CTR Opportunity (7,419 pages) — Bulk Review:** Batch review in cycles of 50 pages sorted by model score.
5. **Monitor & Re-score:** Track feature distributions monthly for drift.

In [6]:
# ── Generate & Export Figures ──────────────────────────────────────────────
fig_dir = '../../work/figures'
os.makedirs(fig_dir, exist_ok=True)

# Figure 1: Score distribution
plt.figure(figsize=(8, 4.5))
plt.hist(lane4['model_score'], bins=50, color='#6c8aff', edgecolor='#1c1e2e', alpha=0.85)
plt.axvline(0.5, color='#f25f5c', linestyle='--', linewidth=2, label='Action Threshold (0.5)')
plt.axvline(base_rate, color='#50fa7b', linestyle=':', linewidth=2, label=f'Base Rate ({base_rate:.2f})')
plt.title('Distribution of Model Scores P(under-CTR) across 22,006 Pages', fontsize=12, fontweight='bold')
plt.xlabel('Model Score P(under-CTR)')
plt.ylabel('Number of Pages')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'score_distribution.png'), dpi=150)
plt.close()

# Figure 2: Reason code distribution
flagged = lane4[lane4['reason_code'] != 'no_action_needed']
rc_counts = flagged['reason_code'].value_counts()

plt.figure(figsize=(9, 4.5))
rc_counts.plot(kind='barh', color='#6c8aff', edgecolor='#1c1e2e')
plt.title('Reason Code Distribution Across Flagged Pages (12,616 Total)', fontsize=12, fontweight='bold')
plt.xlabel('Number of Flagged Pages')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'reason_code_distribution.png'), dpi=150)
plt.close()

print(f"Saved figures to {os.path.abspath(fig_dir)}")

## 7. Artifacts the Paper Embeds

- `work/figures/score_distribution.png` — Score distribution chart embedded in paper Section 4.
- `work/figures/reason_code_distribution.png` — Reason code breakdown chart embedded in paper Section 6.
- `work/outputs/playbook_metrics.json` — Committed metric receipts.

## 8. 5-Minute Showcase Demo Outline (Week-8 Showcase)

*Ready for the Week-8 showcase presentation (5-minute talk structure).* 

---

### ⏱️ Timestamped 5-Minute Slide & Talk Breakdown

#### **0:00 – 1:00 | Minute 1: The Question & Problem**
- **Slide Title:** *Prioritizing Content Refreshes Across 22,000 Pages*
- **The FlyRank Problem:** FlyRank manages thousands of published articles across client portfolios. Currently, production relies on hand-written heuristic flags (e.g. `CTR < 0.5%`), which flag 9,759 pages — creating 195 review cycles for editors operating at 50 pages/cycle. Fixed rules ignore position context (position 3 vs position 18) and flood editors with noisy alerts.
- **Core Question:** *Which visible pages under-capture clicks relative to their position tier, and should be reviewed first for title/snippet metadata or content refresh?*

#### **1:00 – 2:00 | Minute 2: The Method & Data Contract**
- **Slide Title:** *Position-Adjusted Opportunity & Leakage-Free Validation*
- **Data & Slice:** 22,006 working pages from FlyRank's 79-million-row search dataset across 32 clients.
- **Target Proxy:** `is_under_ctr` (1 if page CTR < median CTR of its position tier: top_3, page_1, striking, etc.).
- **Feature Safety:** Strictly excluded `ctr`, `clicks_*`, and `trend_*` to prevent target leakage. Used 16 pre-decision engagement, position, and metadata signals.
- **Validation Design:** 30-client GroupShuffleSplit (8 client portfolios held out completely) to test cross-portfolio generalization.

#### **2:00 – 3:00 | Minute 3: The Chart (Model vs Baseline)**
- **Slide Title:** *Queue Precision: Random Forest vs Transparent Rule Baseline*
- **Visual:** Precision@K Bar Chart (`Figure 1` from paper).
- **Talk Track:** "Here is the head-to-head comparison on the held-out 8-client test set. A fixed rule score achieves Precision@50 = 0.40 — worse than random guessing near the base rate (47.9%). The Random Forest model achieves Precision@50 = 0.76. In an editor's top-50 review queue, 38 out of 50 pages are true under-performers compared to just 20 for the rule — an observed **1.9× lift in queue precision**."

#### **3:00 – 4:00 | Minute 4: One Honest Result & Signal Drivers**
- **Slide Title:** *Honest Validation Metrics & Key Signal Drivers*
- **Metrics:** Random Forest AUC = 0.72 vs Baseline AUC = 0.47 on held-out client portfolios.
- **Surprise Signal:** Permutation importance shows `days_with_sessions` (0.039 AUC drop) and `engagement_rate` (0.022 AUC drop) are the top predictors — demonstrating that on-site user session consistency and engagement carry critical predictive value for search click gaps.
- **Honest Limits:** AUC = 0.72 represents *moderate* discrimination (useful for prioritization, not binary automated actions). Observational study — cannot claim causal CTR lift without A/B testing.

#### **4:00 – 5:00 | Minute 5: One Recommendation & Playbook**
- **Slide Title:** *Ranked Editorial Action Playbook*
- **Top Action:** Prioritize the **342 High-Volume Underperformers** first (pages with >1,000 impressions and high model score) — snippet/title rewrites here yield the highest return per editor hour.
- **Action Queue Distribution:** High-Volume (342) → Position Decay Risk (971) → Low Session Consistency (3,884) → General Review (7,419).
- **Human-in-the-Loop Rule:** Model outputs a decision-support queue with reason codes. Never auto-rewrite without human review. Monitor top features monthly for distribution drift.

## 9. Shareable Cuts

*Drafted for public sharing and employer presentation.* 

---

### 📢 Cut 1: Short Social Post (LinkedIn / X)

> **Beating Fixed Rules in Content Optimization: A Position-Adjusted CTR Model** 🚀
>
> When managing thousands of published web pages, traditional rule flags (like "CTR < 0.5%") flood editorial teams with thousands of low-signal alerts. Position 18 isn't Position 3 — static rules ignore search context.
>
> In my FlyRank Capstone project, I built a machine-learning scoring pipeline on 22,006 search performance records to rank pages by click-through-rate opportunity, adjusting for position tier and weighting by impression volume.
>
> **Key findings:**
> • Evaluated on held-out client portfolios (GroupShuffleSplit), Random Forest achieved **Precision@50 = 0.76** (a **1.9× lift** over the 0.40 rule baseline) and **AUC = 0.72**.
> • On-site session consistency (`days_with_sessions`) and engagement rate proved to be the strongest signals for identifying under-capturing pages.
> • The model outputs a ranked decision-support queue with reason codes, letting editors tackle high-volume title/snippet wins first.
>
> 📄 **Live Deployed Paper:** https://shreeyeshbaral.github.io/ShreeyeshAssignment1/paper/
> 💻 **Code & Notebooks:** https://github.com/shreeyeshbaral/ShreeyeshAssignment1
>
> #MachineLearning #DataScience #SEO #ContentOps #FlyRank #Python

---

### 💼 Cut 2: 3-Sentence Employer-Facing Summary

> **What I built:** I developed a position-adjusted CTR opportunity scoring model and ranked action queue with reason codes to help content teams prioritize high-leverage page metadata and content refreshes.
>
> **On what data:** The model was trained and validated on 22,006 page performance records sampled from FlyRank's 79-million-row search dataset across 32 anonymized clients, using a client-grouped split to rigorously prevent cross-portfolio data leakage.
>
> **What it showed:** On held-out clients, the Random Forest model achieved **Precision@50 = 0.76** (a **1.9× lift** over the 0.40 rule baseline) and **AUC = 0.72**, demonstrating that on-page session consistency and engagement rate provide actionable signal for prioritizing editorial review.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.